# MI Agent 데모

전체 흐름은 다음과 같습니다.


1. 기존 Event 데이터 수 확인
2. Agent 실행
  - 2.1 대상 MI Record 목록 확인
  - 2.2. MetadataCuration 생성확인
  - 2.3. 추가 반영된 Event 목록
  - 2.4. 실행 후 Event 데이터 확인

In [138]:
AGENT_INVOKE_URL = "http://localhost:3002/invoke"

DB_HOST = "127.0.0.1"
DB_PORT = 13306
DB_NAME = "midb"
DB_USER = "root"
DB_PASSWORD = "miagent"

# DB_HOST = "40.82.137.178"
# DB_PORT = 23009
# DB_NAME = "midb"
# DB_USER = "appuser"
# DB_PASSWORD = "aoqdbwjaqSW"

# print("Agent URL:", AGENT_INVOKE_URL)
# print("DB:", f"{DB_HOST}:{DB_PORT}/{DB_NAME}")

request_body = {
    "input": {
        "limit": 5
    },
    "config": {},
    "kwargs": {}
}

import pandas as pd

from sqlalchemy import create_engine, text

def create_db_engine():
    if not DB_PASSWORD:
        raise ValueError("DB_PASSWORD 값을 먼저 입력하세요.")
    return create_engine(
        f"mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}",
        pool_pre_ping=True,
    )

def get_event_count():
    engine = create_db_engine()
    with engine.connect() as conn:
        return conn.execute(text("select count(*) from tmm_mieventm")).scalar_one()

# invoke api
import requests
import json

AUTH_TOKEN = "sk-8895c89d489608766f3b7166ab24ef04"  # env.txt의 API_KEY 값을 넣으세요

headers = {
    "authorization": AUTH_TOKEN,
    "Content-Type": "application/json",
    "aip-user": "test-user"
}

## 1. 기존 Event 데이터 확인

API 실행 전 `tmm_mieventm`에 몇 건이 적재되어 있는지 확인


In [139]:
try:
    before_count = get_event_count()
    print("실행 전 Event 데이터 건수:", before_count)
except Exception as exc:
    before_count = None
    print("DB 사전 확인을 건너뜁니다:", exc)

실행 전 Event 데이터 건수: 0


## 2. Agent 실행

아래 셀은 실제 `/invoke` API를 호출합니다. 실행 중 LLM 호출이 포함되어 있어 시간이 걸릴 수 있습니다.

3.1 ~ 3.3 결과를 내기위한 작업 수행

In [140]:
response = requests.post(
    AGENT_INVOKE_URL,
    json=request_body,
    headers=headers,
    timeout=7200,
)

print("HTTP Status:", response.status_code)
print(response.text[:1000])

response.raise_for_status()
result = response.json()
result.keys()

HTTP Status: 200
{"output":{"summary":"1. 전체 요약 (3줄)\n최근 미국 LNG 수출 플랜트의 피드가스 급증과 Permian 지역 파이프라인 병목 현상, Golden Pass LNG의 첫 수출 등으로 미국 내외 LNG 공급 및 가격 변동성이 크게 확대되고 있다. 동시에 미국-이란 간 긴장과 전쟁, 호르무즈 해협의 지정학적 리스크 심화로 글로벌 LNG 공급망과 가격에 불확실성이 커지고 있다. 미국 정부의 에너지 정책 변화와 인프라 강화도 시장 구조 변화에 영향을 미치고 있다.\n\n2. 핵심 키워드 3개\nLNG 공급망, 지정학적 리스크, 미국 에너지 인프라\n\n3. 중요한 이벤트 2개\n- 미국과 이스라엘의 이란과의 전쟁 및 국방물자생산법 발동으로 글로벌 LNG 가격과 에너지 정책에 직접적 영향 발생\n- Permian 지역 파이프라인 병목으로 Waha Hub 현물가격이 51일 연속 마이너스를 기록하며 미국 내 공급 구조에 심각한 영향을 미침","llm_result":"1. 전체 요약 (3줄)\n최근 미국 LNG 수출 플랜트의 피드가스 급증과 Permian 지역 파이프라인 병목 현상, Golden Pass LNG의 첫 수출 등으로 미국 내외 LNG 공급 및 가격 변동성이 크게 확대되고 있다. 동시에 미국-이란 간 긴장과 전쟁, 호르무즈 해협의 지정학적 리스크 심화로 글로벌 LNG 공급망과 가격에 불확실성이 커지고 있다. 미국 정부의 에너지 정책 변화와 인프라 강화도 시장 구조 변화에 영향을 미치고 있다.\n\n2. 핵심 키워드 3개\nLNG 공급망, 지정학적 리스크, 미국 에너지 인프라\n\n3. 중요한 이벤트 2개\n- 미국과 이스라엘의 이란과의 전쟁 및 국방물자생산법 발동으로 글로벌 LNG 가격과 에너지 정책에 직접적 영향 발생\n- Permian 지역 파이프라인 병목으로 Waha Hub 현물가격이 51일 연속 마이너스를 기록하며 미국 내 공급 구조에 심각한 영향을 미침","rdb_result":[{"mi_event_id

dict_keys(['output', 'metadata'])

## 2.1. 대상 Record 목록

뉴스/리포트 단위의 원천 데이터로, 본문·요약·번역 등의 정보를 포함한다.

In [141]:
limit = 5
record_query = f"""
        select
            r.mi_record_id,
            r.first_created,
            r.record_type,
            r.source_obj_id,
            g.gds_no,
            d.title,
            s.smry,
            i.nws_mrkt_ctgr_cd
        from tmd_mi_record as r
        join tmd_nws_smry as s
            on r.source_obj_id = s.nws_smry_id
           and r.record_type = 'NEWS'
        join tif_refinewsdatar as d
            on s.data_reg_sn = d.data_reg_sn
           and s.first_created = d.first_created
        join tmd_gdsm as g
            on g.gds_id = s.gds_id
        join tmd_nws_sig as i
            on s.nws_smry_id = i.nws_smry_id
        where r.record_type = 'NEWS' and i.nws_mrkt_ctgr_cd = 'LNG' 
        order by r.first_created desc
        limit {limit}
"""
df = pd.read_sql(record_query, create_db_engine())
df

,mi_record_id,first_created,record_type,source_obj_id,gds_no,title,smry,nws_mrkt_ctgr_cd
0,238,2026-04-20 20:57:56,NEWS,198700,70053,UPDATE 1-Trump cites defense production act to...,4월 20일 워싱턴에서 도널드 트럼프 미국 대통령은 국방물자생산법(DPA)을 발동해...,LNG
1,237,2026-04-20 18:58:03,NEWS,198497,70053,UPDATE 1-US natgas hits one-week high as outpu...,4월 20일 미국 천연가스 선물은 최근 2주간 생산 감소와 5월 초까지 더 서늘한 ...,LNG
2,236,2026-04-20 18:43:43,NEWS,198490,70053,ANALYSIS-Gulf worries US-Iran talks may cement...,걸프 지역 당국자와 분석가들은 이슬라마바드에서 이어질 것으로 예상되는 미-이란 협상...,LNG
3,235,2026-04-20 18:30:56,NEWS,198481,70053,UPDATE 1-Tanker arrives at Golden Pass Texas f...,골든패스 LNG 수출 시설(텍사스)에 첫 수출 화물을 싣기 위한 LNG 탱커 ‘알 ...,LNG
4,231,2026-04-20 17:47:04,NEWS,198449,70053,WRAPUP 8-Fate of Iran peace talks uncertain as...,로이터에 따르면 이란은 4월 7일 발표된 2주 휴전의 종료 시한이 임박한 가운데 파...,LNG


## 2.2. MetadataCuration 생성확인 (Event 생성 및 매핑 품질을 위한 중간 데이터 생성)

In [142]:
import pandas as pd
import json

# JSON 파일 경로
ra_file_path = '../../docs/demos/record_analysis.json'

# 파일 읽기
with open(ra_file_path, 'r', encoding='utf-8') as f:
    data = json.load(f)

# 데이터 펼치기
rows = []
for cat in data["categories"]:
    row = {
        "level_1": cat["level_1"],
        "level_2": cat["level_2"],
        "entities": ", ".join([e["name"] for e in cat["entities"]]),
        "entity_types": ", ".join([e["entity_type"] for e in cat["entities"]]),
        "occurrence_time": cat["event_timing"]["occurrence_time"],
        "time_precision": cat["event_timing"]["occurrence_time_precision"],
        "regions": ", ".join(cat["regions"]),
        "supply_demand_side": cat["supply_demand_side"],
        "score": cat["score"],
        "reason": cat["reason"]
    }
    rows.append(row)

# DataFrame 생성
ra_df = pd.DataFrame(rows)

Record를 기반으로 Event 생성 및 연결을 지원하는 보조 분석 데이터를 포함한다.

(Record를 분석해 이벤트 유형 후보, 엔티티, 발생 시점/지역, 연관도(score), 근거(reason) 등의 메타데이터를 추출하고, 이를 구조화하여 Event 생성·매핑에 활용)

In [143]:
# 출력
ra_df

,level_1,level_2,entities,entity_types,occurrence_time,time_precision,regions,supply_demand_side,score,reason
0,geopolitical_risk,geopolitical_tension_instability,"Iran, United States, Pakistan, China, Hormuz S...","country, country, country, country, region",2024-04-21,day,"Middle East, Hormuz Strait",supply,9.0,"이란과 미국 간의 긴장, 호르무즈 해협의 정상 통항 촉구, 항만 봉쇄 및 선박 나포..."
1,logistics_shipping,cargo_flow_change_port_canal_disruption,"Hormuz Strait, Iran, United States","region, country, country",2024-04-21,day,"Middle East, Hormuz Strait",supply,8.0,"미국의 이란 항만 봉쇄 해제 노력, 선박 나포 주장, 중국의 정상 통항 촉구 등은 ..."


## 2.3. 추가 반영된 Event 목록

`read_rdb` 노드에서 DB를 다시 조회한 결과입니다. 이 값이 최종 요약 LLM의 입력으로 사용됩니다.


In [144]:
output = result["output"]
events_df = pd.DataFrame(output.get("rdb_result", []))

display_columns = [
    "mi_event_id",
    "first_created",
    "event_title_ko",
    "one_line_summary",
    "level_1_code",
    "level_2_code",
    "occurred_at",
]

existing_columns = [col for col in display_columns if col in events_df.columns]

여러 Record에서 공통적으로 식별되는 시장 사건/이슈를 하나의 단위로 추출된 결과이다.

In [145]:
events_df[existing_columns] if not events_df.empty else events_df

,mi_event_id,first_created,event_title_ko,one_line_summary,level_1_code,level_2_code,occurred_at
0,58,2026-04-29T06:33:24,미국 LNG 수출 플랜트 피드가스 급증 및 지역 수요 경쟁 심화,미국 LNG 수출 플랜트로의 피드가스 흐름이 기록적 수준에 근접하며 수출 수요 급증...,supply_demand_balance,regional_demand_competition,2026-04-20T18:58:03
1,57,2026-04-29T06:33:24,Permian 지역 파이프라인 병목 및 Waha Hub 현물가격 마이너스 지속,Permian 지역 파이프라인 제약으로 Waha Hub 현물가격이 51일 연속 마이...,infrastructure_facility,pipeline_transport_infrastructure_capacity_uti...,2026-04-20T18:58:03
2,56,2026-04-29T06:33:24,미국 Golden Pass LNG 첫 수출 화물 출항 및 설비 가동 확대,Golden Pass LNG의 첫 수출 화물 출항으로 미국 LNG 공급 확대 신호탄...,infrastructure_facility,liquefaction_terminal_facility_capacity_utiliz...,2026-04-20T20:57:56
3,55,2026-04-29T06:33:24,미국과 이스라엘의 이란과의 전쟁 및 에너지 관련 국방물자생산법 발동,미국과 이스라엘의 이란과의 전쟁 및 국방물자생산법 발동으로 에너지 공급과 수요에 변...,geopolitical_risk,war_military_conflict,2026-04-20T20:57:56
4,54,2026-04-29T06:33:23,미국-이란 긴장 및 호르무즈 해협 지정학적 리스크 심화,미국과 이란 간 긴장 고조 및 호르무즈 해협 통항 불확실성으로 LNG 공급망에 지정...,geopolitical_risk,geopolitical_tension_instability,2026-04-20T18:43:43


## 2.4. 실행 후 Event 데이터 확인

API 실행 후 `tmm_mieventm` 테이블에 row가 증가했는지 확인합니다.


In [146]:
try:
    after_count = get_event_count()
    print("실행 전 건수:", before_count)
    print("실행 후 건수:", after_count)
    if before_count is not None:
        print("증가 건수:", after_count - before_count)
except Exception as exc:
    after_count = None
    print("DB 사후 확인을 건너뜁니다:", exc)


실행 전 건수: 0
실행 후 건수: 5
증가 건수: 5
